In [4]:
# Install Qiskit
!pip install qiskit qiskit-aer matplotlib

# Import the tools we need
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt

print("Qiskit is installed and ready!")

Qiskit is installed and ready!


In [5]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# 1. Define Processor A (5 qubits) and Processor B (9 qubits) coupling maps
processor_A_coupling = [[0, 1], [1, 2], [1, 3], [3, 4]]
processor_B_coupling = [
    [0, 1], [1, 2], [1, 6], [2, 3], [2, 6], [2, 5],
    [3, 4], [4, 5], [5, 6], [6, 7], [6, 8], [8, 0]
]

# 2. Build a 5-vertex Max-Cut QAOA circuit to fit Processor A's capacity
q_nodes = 5
qaoa_circuit = QuantumCircuit(q_nodes)

# Apply initial Superposition (Hadamards)
for i in range(q_nodes):
    qaoa_circuit.h(i)

# Apply cost layer interactions (CX and Rz gates)
edges = [(0, 1), (1, 2), (2, 3), (3, 4)]
for edge in edges:
    qaoa_circuit.cx(edge[0], edge[1])
    qaoa_circuit.rz(0.5, edge[1])
    qaoa_circuit.cx(edge[0], edge[1])

qaoa_circuit.measure_all()

# 3. Transpile and execute on Processor A
sim = AerSimulator()
compiled_A = transpile(qaoa_circuit, coupling_map=processor_A_coupling, optimization_level=1, seed_transpiler=42)
result_A = sim.run(compiled_A, shots=4096).result()

# 4. Transpile and execute on Processor B
compiled_B = transpile(qaoa_circuit, coupling_map=processor_B_coupling, optimization_level=1, seed_transpiler=42)
result_B = sim.run(compiled_B, shots=4096).result()

# 5. Print Comparison Metrics
print("=== MAX-CUT QAOA BENCHMARK RESULTS ===")
print("\n--- Processor A (5 Qubits - Line/Branch) ---")
print(f"Circuit Depth: {compiled_A.depth()}")
print(f"Total Operations: {compiled_A.count_ops()}")

print("\n--- Processor B (9 Qubits - Heavy-Hex) ---")
print(f"Circuit Depth: {compiled_B.depth()}")
print(f"Total Operations: {compiled_B.count_ops()}")

=== MAX-CUT QAOA BENCHMARK RESULTS ===

--- Processor A (5 Qubits - Line/Branch) ---
Circuit Depth: 17
Total Operations: OrderedDict({'h': 9, 'cx': 8, 'measure': 5, 'rz': 3, 'swap': 1, 'u3': 1, 'barrier': 1})

--- Processor B (9 Qubits - Heavy-Hex) ---
Circuit Depth: 14
Total Operations: OrderedDict({'cx': 8, 'h': 5, 'measure': 5, 'rz': 4, 'barrier': 1})
